# RNA-seq exploratory notebook (`rna_seq.ipynb`)

This notebook analyzes **existing Salmon outputs** from the RNA-seq GCP tutorial.

It expects these files to already exist (generated by `notebooks/rna_seq_gcp/04_run_pipeline.md`):

- `results/salmon_SRR453566/quant.sf`
- `results/salmon_SRR453567/quant.sf`
- `results/tx2gene.tsv`

> ✅ This notebook does **not** run alignment or quantification. It only loads and analyzes previously generated files.

## 0) Imports and plotting settings

We will use only beginner-friendly Python libraries:
- `pandas`, `numpy` for data handling
- `matplotlib`, `seaborn` for plotting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)

## 1) Explain the structure of `quant.sf`

Each Salmon `quant.sf` file is a transcript-level expression table with these columns:

- `Name`: transcript ID
- `Length`: transcript length
- `EffectiveLength`: effective transcript length after bias correction
- `TPM`: normalized abundance (*Transcripts Per Million*)
- `NumReads`: estimated number of reads assigned to the transcript

For this notebook, we focus on **TPM** values for simple cross-sample comparison.

## 2) Load both samples and merge into one transcript-level table

In [ ]:
# Update this path if needed (for example, if your results are under ~/rna_seq_tutorial/results)
results_dir = Path("results")

sample_files = {
    "SRR453566": results_dir / "salmon_SRR453566" / "quant.sf",
    "SRR453567": results_dir / "salmon_SRR453567" / "quant.sf",
}

tx2gene_path = results_dir / "tx2gene.tsv"

for sample, path in sample_files.items():
    print(sample, path, "exists:", path.exists())
print("tx2gene", tx2gene_path, "exists:", tx2gene_path.exists())

In [ ]:
q1 = pd.read_csv(sample_files["SRR453566"], sep="\t")
q2 = pd.read_csv(sample_files["SRR453567"], sep="\t")

q1 = q1[["Name", "TPM"]].rename(columns={"TPM": "TPM_SRR453566"})
q2 = q2[["Name", "TPM"]].rename(columns={"TPM": "TPM_SRR453567"})

tx_expr = q1.merge(q2, on="Name", how="inner")

print("Transcript table shape:", tx_expr.shape)
tx_expr.head()

## 3) Aggregate transcript TPM to gene-level using `tx2gene.tsv`

`tx2gene.tsv` is a two-column mapping:
1. transcript ID
2. gene ID

We join transcript TPM values to gene IDs, then sum transcript TPM within each gene.

In [ ]:
tx2gene = pd.read_csv(tx2gene_path, sep="\t", header=None, names=["Name", "gene_id"])
print("tx2gene shape:", tx2gene.shape)
tx2gene.head()

In [ ]:
tx_annot = tx_expr.merge(tx2gene, on="Name", how="left")

missing_gene = tx_annot["gene_id"].isna().mean()
print(f"Fraction of transcripts without gene_id: {missing_gene:.3f}")

gene_expr = (
    tx_annot.dropna(subset=["gene_id"])
    .groupby("gene_id", as_index=False)[["TPM_SRR453566", "TPM_SRR453567"]]
    .sum()
)

print("Gene table shape:", gene_expr.shape)
gene_expr.head()

## 4) Log-transform expression values

TPM values are often right-skewed, so we use `log2(TPM + 1)` for visualization.

In [ ]:
gene_expr["log2TPM_SRR453566"] = np.log2(gene_expr["TPM_SRR453566"] + 1)
gene_expr["log2TPM_SRR453567"] = np.log2(gene_expr["TPM_SRR453567"] + 1)

gene_expr[["gene_id", "log2TPM_SRR453566", "log2TPM_SRR453567"]].head()

## 5) Visualization: Histogram of gene-level expression values

In [ ]:
plt.figure()
plt.hist(gene_expr["log2TPM_SRR453566"], bins=40, alpha=0.6, label="SRR453566")
plt.hist(gene_expr["log2TPM_SRR453567"], bins=40, alpha=0.6, label="SRR453567")
plt.xlabel("log2(TPM + 1)")
plt.ylabel("Number of genes")
plt.title("Histogram of gene-level expression")
plt.legend()
plt.show()

## 6) Visualization: Scatter plot comparing gene expression between samples

In [ ]:
plt.figure()
plt.scatter(
    gene_expr["log2TPM_SRR453566"],
    gene_expr["log2TPM_SRR453567"],
    s=10,
    alpha=0.5
)
plt.xlabel("SRR453566 log2(TPM + 1)")
plt.ylabel("SRR453567 log2(TPM + 1)")
plt.title("Gene expression scatter plot")

mn = min(gene_expr["log2TPM_SRR453566"].min(), gene_expr["log2TPM_SRR453567"].min())
mx = max(gene_expr["log2TPM_SRR453566"].max(), gene_expr["log2TPM_SRR453567"].max())
plt.plot([mn, mx], [mn, mx], linestyle="--", color="red")

plt.show()

## 7) Visualization: PCA plot (demonstration only)

With only two samples, PCA is very limited. This plot is included only to show the workflow.

In [ ]:
X = np.vstack([
    gene_expr["log2TPM_SRR453566"].values,
    gene_expr["log2TPM_SRR453567"].values,
])

X_centered = X - X.mean(axis=0, keepdims=True)
U, S, VT = np.linalg.svd(X_centered, full_matrices=False)
pcs = U * S

pca_df = pd.DataFrame({
    "sample": ["SRR453566", "SRR453567"],
    "PC1": pcs[:, 0],
    "PC2": pcs[:, 1] if pcs.shape[1] > 1 else 0,
})

plt.figure()
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="sample", s=120)
for _, r in pca_df.iterrows():
    plt.text(r["PC1"], r["PC2"], " " + r["sample"], va="center")
plt.title("PCA of two RNA-seq samples (demo)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

## 8) Visualization: Heatmap of top variable genes

In [ ]:
gene_expr["variance"] = gene_expr[["log2TPM_SRR453566", "log2TPM_SRR453567"]].var(axis=1)
top_n = 30
top_var = gene_expr.nlargest(top_n, "variance")

heatmap_data = top_var.set_index("gene_id")[["log2TPM_SRR453566", "log2TPM_SRR453567"]]

plt.figure(figsize=(6, 10))
sns.heatmap(heatmap_data, cmap="viridis")
plt.title(f"Top {top_n} variable genes (log2 TPM)")
plt.xlabel("Sample")
plt.ylabel("Gene ID")
plt.show()

## 9) Visualization: Volcano plot (log2 fold-change only, no p-values)

In [ ]:
gene_expr["log2FC_567_vs_566"] = (
    np.log2(gene_expr["TPM_SRR453567"] + 1) - np.log2(gene_expr["TPM_SRR453566"] + 1)
)

gene_expr["pseudo_significance"] = -np.log10(1 / (np.abs(gene_expr["log2FC_567_vs_566"]) + 1.1))

plt.figure()
plt.scatter(
    gene_expr["log2FC_567_vs_566"],
    gene_expr["pseudo_significance"],
    s=10,
    alpha=0.5
)
plt.axvline(1, linestyle="--", color="red")
plt.axvline(-1, linestyle="--", color="red")
plt.xlabel("log2 fold change (SRR453567 vs SRR453566)")
plt.ylabel("Demonstration y-axis (not a real p-value)")
plt.title("Volcano-style plot (demonstration only)")
plt.show()

## 10) Important limitations

Because this dataset has only **two samples and no biological replicates**:

- We can do **exploratory visualization** only.
- We cannot estimate biological variability reliably.
- We cannot make strong differential-expression claims.
- The volcano plot above is **illustrative only** and does not include valid statistical testing.

For real differential expression studies, use replicated experimental designs and dedicated methods (outside the scope of this beginner notebook).